<a href="https://colab.research.google.com/github/mohammad-shahwan/Book-Extractor/blob/main/Books.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Extracting the names of books

In [ ]:
import requests
from bs4 import BeautifulSoup

def get_book_titles_with_metadata(libgen_search_url):
    response = requests.get(libgen_search_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    book_titles = []

    links = soup.find_all('a', href=True, id=True)

    for link in links:
        title_text = link.contents[0].strip() if link.contents else None

        if title_text:
            book_titles.append(title_text)

    return book_titles

libgen_search_url = 'https://libgen.is/search.php?req=Entrepreneurship&lg_topic=libgen&open=0&view=simple&res=25&phrase=1&column=def'
book_titles = get_book_titles_with_metadata(libgen_search_url)

for i, title in enumerate(book_titles, 1):
    print(f"{i}. {title}")


1. From Concept to Wall Street: A Complete Guide to Entrepreneurship and Venture Capital
2. Entrepreneurship and economic growth
3. Pioneers in Entrepreneurship and Small Business Research
4. Werbemanagement in jungen Wachstumsunternehmen : eine Analyse des Einflusses der Werbeplanung und -kontrolle auf den Erfolg junger Wachstumsunternehmen unter Berücksichtigung der Unternehmensentwicklung
5. Business Idea The Early Stages of Entrepreneurship
6. Corporate Entrepreneurship and Venturing
7. Pioneers in Entrepreneurship and Small Business Research
8. Pioneers in Entrepreneurship and Small Business Research
9. The Role of Labour Mobility and Informal Networks for Knowledge Transfer
10. Entrepreneurship In The United States: Future is Now
11. Understanding the Financial Score
12. Public Policy and the Economics of Entrepreneurship
13. The ABCs of Entrepreneurship: Entrepreneurial Professors From the World's Top Business Schools on the Fundamentals All Business Professionals Should Know A

Names with ratings of the books and the number of people that rated the book

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote

base_url = "https://www.goodreads.com/search?utf8=%E2%9C%93&q={}&search_type=books&search%5Bfield%5D=on"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3",
    "Accept-Language": "en-US,en;q=0.9",
    "Accept-Encoding": "gzip, deflate, br",
    "Connection": "keep-alive"
}
c=1
bookwithhtml={}
for book_title in book_titles:
    search_url = base_url.format(quote(book_title))

    response = requests.get(search_url, headers=headers)

    if response.status_code == 200:
      soup = BeautifulSoup(response.content, 'html.parser')

      html_content = soup.prettify()
      bookwithhtml[book_title]=html_content
      print(f"HTML content for '{book_title}':\n")
      print(c)
      c+=1
    else:
        print(f"Failed to retrieve the page for '{book_title}' (Status code: {response.status_code})")


HTML content for 'From Concept to Wall Street: A Complete Guide to Entrepreneurship and Venture Capital':

1
HTML content for 'Entrepreneurship and economic growth':

2
HTML content for 'Pioneers in Entrepreneurship and Small Business Research':

3
HTML content for 'Werbemanagement in jungen Wachstumsunternehmen : eine Analyse des Einflusses der Werbeplanung und -kontrolle auf den Erfolg junger Wachstumsunternehmen unter Berücksichtigung der Unternehmensentwicklung':

4
HTML content for 'Business Idea The Early Stages of Entrepreneurship':

5
HTML content for 'Corporate Entrepreneurship and Venturing':

6
HTML content for 'Pioneers in Entrepreneurship and Small Business Research':

7
HTML content for 'Pioneers in Entrepreneurship and Small Business Research':

8
HTML content for 'The Role of Labour Mobility and Informal Networks for Knowledge Transfer':

9
HTML content for 'Entrepreneurship In The United States: Future is Now':

10
HTML content for 'Understanding the Financial Score':

In [ ]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import quote
import re

def get_book_titles_with_metadata(libgen_search_url):
    response = requests.get(libgen_search_url)
    soup = BeautifulSoup(response.text, 'html.parser')

    book_titles = []
    links = soup.find_all('a', href=True, id=True)

    for link in links:
        title_text = link.contents[0].strip() if link.contents else None

        if title_text:
            book_titles.append(title_text)

    return book_titles

def get_goodreads_mini_rating(book_title):
    base_url = "https://www.goodreads.com/search?utf8=%E2%9C%93&q={}&search_type=books&search%5Bfield%5D=on"
    search_url = base_url.format(quote(book_title))

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3",
        "Accept-Language": "en-US,en;q=0.9",
        "Accept-Encoding": "gzip, deflate, br",
        "Connection": "keep-alive"
    }

    response = requests.get(search_url, headers=headers)

    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        mini_rating = soup.find('span', class_='minirating')

        if mini_rating:
            rating_text = mini_rating.text.strip()

            match = re.search(r'(\d\.\d+) avg rating — ([\d,]+) ratings', rating_text)
            if match:
                avg_rating = float(match.group(1))
                num_ratings = int(match.group(2).replace(",", ""))
                return avg_rating, num_ratings
            else:
                return None, None
        else:
            return None, None
    else:
        return None, None

def main():
    libgen_search_url = 'https://libgen.is/search.php?req=Entrepreneurship&lg_topic=libgen&open=0&view=simple&res=25&phrase=1&column=def'
    book_titles = get_book_titles_with_metadata(libgen_search_url)

    valid_books = []

    for title in book_titles:
        print(f"Searching for: {title}")
        avg_rating, num_ratings = get_goodreads_mini_rating(title)

        if avg_rating is not None and num_ratings > 0:
            valid_books.append((title, avg_rating, num_ratings))
            print(f"Added '{title}' with rating {avg_rating} from {num_ratings} ratings")
        else:
            print(f"Skipped '{title}' due to no rating or invalid rating")

    valid_books_sorted = sorted(valid_books, key=lambda x: (-x[1], -x[2]))

    print("\nTop Books by Rating:")
    for i, (title, avg_rating, num_ratings) in enumerate(valid_books_sorted, 1):
        print(f"{i}. {title} - {avg_rating} average rating from {num_ratings} ratings")

    return valid_books_sorted

if __name__ == "__main__":
    top_books = main()


Searching for: From Concept to Wall Street: A Complete Guide to Entrepreneurship and Venture Capital
Added 'From Concept to Wall Street: A Complete Guide to Entrepreneurship and Venture Capital' with rating 3.57 from 7 ratings
Searching for: Entrepreneurship and economic growth
Added 'Entrepreneurship and economic growth' with rating 4.1 from 10 ratings
Searching for: Pioneers in Entrepreneurship and Small Business Research
Added 'Pioneers in Entrepreneurship and Small Business Research' with rating 5.0 from 2 ratings
Searching for: Werbemanagement in jungen Wachstumsunternehmen : eine Analyse des Einflusses der Werbeplanung und -kontrolle auf den Erfolg junger Wachstumsunternehmen unter Berücksichtigung der Unternehmensentwicklung
Skipped 'Werbemanagement in jungen Wachstumsunternehmen : eine Analyse des Einflusses der Werbeplanung und -kontrolle auf den Erfolg junger Wachstumsunternehmen unter Berücksichtigung der Unternehmensentwicklung' due to no rating or invalid rating
Searchin

In [ ]:
import requests
from bs4 import BeautifulSoup

keyword = input("Enter book genre: ").split()
keyword="+".join(keyword)
url = f"https://libgen.is/search.php?&req={keyword}&phrase=1&view=simple&column=def&sort=def&sortmode=ASC&page=1"

response = requests.get(url)

if response.status_code == 200:
  html_content = response.content
  soup = BeautifulSoup(html_content, 'html.parser')
  extracted_html = soup.prettify()
  print("Success!")
else:
  print(f"Fail: {response.status_code}")


KeyboardInterrupt: Interrupted by user

In [ ]:
all_book_links=[]
pagenum=int(input("How many pages: "))
for i in range(1,pagenum+1):
  url = f"https://libgen.is/search.php?&req={keyword}&phrase=1&view=simple&column=def&sort=def&sortmode=ASC&page={i}"
  response = requests.get(url)
  html_content = response.content
  soup = BeautifulSoup(html_content, 'html.parser')
  pattern = "http://libgen.li/ads.php?md5="
  matching_links = []
  for link in soup.find_all('a', href=True):
    if link['href'].startswith(pattern):
      matching_links.append(link['href'])
  page_book_links=[]
  for link in matching_links:
    page_book_links.append(link)
  all_book_links.append(page_book_links)

In [ ]:
all_book_links

In [ ]:
for i in range (len(all_book_links)):
  print(len(all_book_links[i]))

In [ ]:
import os
import requests
from bs4 import BeautifulSoup
from requests.exceptions import ChunkedEncodingError, ConnectionError

output_directory = "/content/collection_of_books"
os.makedirs(output_directory, exist_ok=True)

get_pattern = "get.php?md5="

def download_file(url, file_path):
    try:
        with requests.get(url, stream=True) as download_response:
            download_response.raise_for_status()
            with open(file_path, 'wb') as file:
                for chunk in download_response.iter_content(chunk_size=8192):
                    if chunk:
                        file.write(chunk)
        return True
    except (ChunkedEncodingError, ConnectionError, requests.exceptions.RequestException) as e:
        print(f"    Failed to download {url}. Error: {e}")
        return False

for page_index, page_links in enumerate(all_book_links, start=1):
    print(f"Processing page {page_index}...")

    for book_index, book_link in enumerate(page_links, start=1):
        print(f"  Fetching book {book_index} from {book_link}...")

        try:
            response = requests.get(book_link)
            response.raise_for_status()
        except requests.exceptions.RequestException as e:
            print(f"    Failed to access {book_link}. Error: {e}")
            continue

        soup = BeautifulSoup(response.content, 'html.parser')

        get_link_tag = soup.find('a', href=True, string='GET')
        if get_link_tag and get_pattern in get_link_tag['href']:
            download_link = book_link.split("ads.php")[0] + get_link_tag['href']

            file_name = download_link.split('md5=')[-1] + ".pdf"
            file_path = os.path.join(output_directory, file_name)

            print(f"    Downloading file from {download_link}...")
            if download_file(download_link, file_path):
                print(f"    File saved as {file_name} in '{output_directory}' folder.")
            else:
                print(f"    Retrying download for {download_link}...")
                download_file(download_link, file_path)
        else:
            print(f"    'GET' link not found in {book_link}.")
